# Barren Plateau Phenomenon: Global vs. Local Cost Functions
## Interactive Simulator with Qubit Scaling & Cost Landscape Analysis

---

### Key Theoretical Background:
1. **Global Cost Function ($C_{\text{Global}}$)**:
   * Defined over all $n$ qubits simultaneously: $C_G = \langle 0^{\otimes n} | U^\dagger H_G U | 0^{\otimes n} \rangle$.
   * As proven by **McClean et al. (2018)**:
     $$\operatorname{Var}\left[\frac{\partial C_G}{\partial \theta}\right] \in \mathcal{O}(2^{-n})$$
   * The cost landscape flattens **exponentially** in the number of qubits $n$, rendering optimization virtually impossible for $n \ge 6$.

2. **Local Cost Function ($C_{\text{Local}}$)**:
   * Defined as a sum of few-body (1-qubit or 2-qubit) local observables: $C_L = \frac{1}{n} \sum_{i=1}^n \langle U^\dagger H_i U \rangle$.
   * As proven by **Cerezo et al. (2021)**:
     $$\operatorname{Var}\left[\frac{\partial C_L}{\partial \theta}\right] \in \Omega\left(\frac{1}{\text{poly}(n)}\right)$$
   * For shallow circuits, the gradient variance decays only **polynomially** $\sim \mathcal{O}(1/n)$, preserving trainability even at higher qubit counts!


In [1]:
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

os.makedirs('figures', exist_ok=True)

qubit_list = [2, 4, 6, 8, 10, 12, 14]
theta = np.linspace(-np.pi, np.pi, 200)

def get_variance(n, mode):
    if mode == 'global':
        return 0.25 * (2.0 ** (-(n - 2)))
    else:
        return 0.25 * ((2.0 / n) ** 1.45)

def get_landscape(n, mode):
    amp = (2.0 ** (-(n - 2) * 0.48)) if mode == 'global' else ((2.0 / n) ** 0.55)
    w1 = 0.30 * (1.0 - np.cos(theta))
    w2 = 0.26 * (np.abs(np.sin(1.8 * theta)) ** 1.4)
    w3 = 0.08 * np.cos(3.2 * theta)
    base_c = w1 + w2 + w3
    mask = np.abs(theta) < 0.22
    base_c[mask] *= (np.abs(theta[mask]) / 0.22) ** 1.6
    c0 = 0.48
    return c0 + (base_c - c0) * amp

fig = make_subplots(
    rows=2, cols=1,
    vertical_spacing=0.18,
    row_heights=[0.48, 0.52],
    subplot_titles=(
        "<b>1D Cost Landscape Slice C(θ) [Global: O(2⁻ⁿ)]</b>",
        "<b>Gradient Variance Var(∂C/∂θ) vs Number of Qubits (n)</b>"
    )
)

modes = ['global', 'local']
for mode in modes:
    for n in qubit_list:
        c_curve = get_landscape(n, mode)
        var_val = get_variance(n, mode)
        
        # 1. Landscape Line
        fig.add_trace(go.Scatter(
            x=theta, y=c_curve, mode='lines',
            line=dict(color='#2563EB', width=2.8),
            fill='tozeroy', fillcolor='rgba(59, 130, 246, 0.15)',
            name='Cost Landscape', visible=False
        ), row=1, col=1)
        
        # 2. Probe Point
        probe_idx = int(len(theta) * 0.68)
        fig.add_trace(go.Scatter(
            x=[theta[probe_idx]], y=[c_curve[probe_idx]], mode='markers',
            marker=dict(size=8, color='#2563EB', line=dict(color='white', width=1.5)),
            name='Probe Point', visible=False
        ), row=1, col=1)
        
        # 3. Local Curve
        local_vars = [get_variance(q, 'local') for q in qubit_list]
        fig.add_trace(go.Scatter(
            x=qubit_list, y=local_vars, mode='lines+markers',
            line=dict(color='#16A34A', width=2.0, dash='dash'),
            marker=dict(size=6, color='#16A34A'),
            name='Local Cost: O(1/poly(n))', visible=False
        ), row=2, col=1)
        
        # 4. Global Curve
        global_vars = [get_variance(q, 'global') for q in qubit_list]
        fig.add_trace(go.Scatter(
            x=qubit_list, y=global_vars, mode='lines+markers',
            line=dict(color='#2563EB', width=2.0, dash='dash'),
            marker=dict(size=6, color='#2563EB'),
            name='Global Cost: O(2^-n)', visible=False
        ), row=2, col=1)
        
        # 5. Active Marker
        fig.add_trace(go.Scatter(
            x=[n], y=[var_val], mode='markers',
            marker=dict(size=12, color='#38BDF8', line=dict(color='#0F172A', width=2.5)),
            name=f'Active (n={n})', visible=False
        ), row=2, col=1)

# Set initial visible: Global, n=2
for i in range(5):
    fig.data[i].visible = True

def get_visibility_mask(target_mode, target_n):
    mask = [False] * len(fig.data)
    m_idx = 0 if target_mode == 'global' else 1
    n_idx = qubit_list.index(target_n)
    start_idx = (m_idx * len(qubit_list) + n_idx) * 5
    for i in range(5):
        mask[start_idx + i] = True
    return mask

buttons = []
for m in modes:
    m_label = "Global Cost" if m == 'global' else "Local Cost"
    mask = get_visibility_mask(m, 2)
    var2 = get_variance(2, m)
    slope2 = np.sqrt(var2)
    buttons.append(dict(
        label=m_label, method="update",
        args=[
            {"visible": mask},
            {
                "title": f"<b>Barren Plateau Phenomenon</b><br><span style='font-size:12px;color:#475569;'>Cost Type: <b>{m.upper()}</b> | Current n = 2 | Variance Σ² = {var2:.4f} | Slope = {slope2:.3f}</span>",
                "annotations[0].text": f"<b>1D Cost Landscape Slice C(θ) [{m.capitalize()}: {'O(2⁻ⁿ)' if m=='global' else 'O(1/poly(n))'}]</b>"
            }
        ]
    ))

slider_steps = []
for n in qubit_list:
    mask_glob = get_visibility_mask('global', n)
    var_g = get_variance(n, 'global')
    slope_g = np.sqrt(var_g)
    slider_steps.append(dict(
        label=str(n), method="update",
        args=[
            {"visible": mask_glob},
            {
                "title": f"<b>Barren Plateau Phenomenon</b><br><span style='font-size:12px;color:#475569;'>Current n = <b>{n} qubits</b> | Variance Σ² = {var_g:.4f} | Landscape Slope = {slope_g:.3f}</span>"
            }
        ]
    ))

sliders = [dict(
    active=0,
    currentvalue=dict(prefix="Number of Qubits (n): ", font=dict(size=13, color="#0F172A"), visible=True),
    pad=dict(t=35, b=10),
    steps=slider_steps,
    x=0.08, y=-0.08, len=0.84
)]

updatemenus = [dict(
    type="buttons", direction="right",
    x=0.5, y=1.14, xanchor="center",
    buttons=buttons,
    bgcolor="#F1F5F9", bordercolor="#CBD5E1",
    font=dict(size=12, color="#0F172A")
)]

fig.update_layout(
    height=800, template='plotly_white',
    title=dict(
        text="<b>Barren Plateau Phenomenon</b><br><span style='font-size:12px;color:#475569;'>Current n = <b>2 qubits</b> | Variance Σ² = 0.2500 | Landscape Slope = 0.500</span>",
        x=0.05, y=0.98
    ),
    updatemenus=updatemenus, sliders=sliders, showlegend=False, hovermode='closest'
)

fig.update_xaxes(title_text="<b>Parameter θ</b>", tickvals=[-np.pi, 0, np.pi], ticktext=["-π", "0", "π"], range=[-np.pi, np.pi], row=1, col=1)
fig.update_yaxes(title_text="<b>Cost C(θ)</b>", range=[-0.05, 1.05], tickvals=[0.0, 0.5, 1.0], row=1, col=1)

fig.update_xaxes(title_text="<b>Number of Qubits (n) →</b>", tickvals=qubit_list, range=[1.5, 14.5], row=2, col=1)
fig.update_yaxes(type="log", title_text="<b>Gradient Variance Var(∂C/∂θ) ↑</b>",
                 tickvals=[1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
                 ticktext=["10µ", "100µ", "1m", "10m", "100m", "1"], range=[-5.1, 0.1], row=2, col=1)

fig.write_html('figures/fig_barren_plateau_plotly.html', include_plotlyjs=True)
fig.show()


### Standalone Interactive UI Widget (Live Slider & Toggle)

In [2]:
# Display the Pixel-Perfect Interactive Web Widget directly inside the Notebook:
from IPython.display import IFrame
IFrame(src='../figures/barren_plateau_interactive.html', width='100%', height=720)
